In [1]:
from pathlib import Path
import os
import json
import csv
import gzip
import zipfile
import textwrap
import traceback
from datetime import datetime

import pandas as pd

ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2")

OUT_DIR = Path.cwd() / "kkt_data_audit_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_TXT = OUT_DIR / "kkt_file_audit_report.txt"
FILES_CSV = OUT_DIR / "kkt_files_inventory.csv"
SAMPLES_DIR = OUT_DIR / "samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("EXISTS:", ROOT.exists())
print("OUT_DIR:", OUT_DIR)

if not ROOT.exists():
    raise FileNotFoundError(ROOT)

ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2
EXISTS: True
OUT_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data_audit_outputs


In [3]:
# ============================================================
# Helpers
# ============================================================

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def size_mb(path: Path) -> float:
    return path.stat().st_size / 1024 / 1024


def safe_name(s: str) -> str:
    s = str(s)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        s = s.replace(b, "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")


def write_report(line=""):
    with open(REPORT_TXT, "a", encoding="utf-8") as f:
        f.write(str(line) + "\n")


def detect_ext(path: Path) -> str:
    name = path.name.lower()
    if name.endswith(".csv.gz"):
        return ".csv.gz"
    if name.endswith(".jsonl.gz"):
        return ".jsonl.gz"
    if name.endswith(".json.gz"):
        return ".json.gz"
    if name.endswith(".txt.gz"):
        return ".txt.gz"
    return path.suffix.lower() if path.suffix else "[no_ext]"


def list_files(root: Path):
    rows = []
    for p in root.rglob("*"):
        if p.is_file():
            rows.append({
                "path": str(p),
                "relative_path": str(p.relative_to(root)),
                "parent": str(p.parent.relative_to(root)),
                "name": p.name,
                "ext": detect_ext(p),
                "size_mb": round(size_mb(p), 3),
                "size_bytes": p.stat().st_size,
                "modified": datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            })
    return pd.DataFrame(rows)


files_df = list_files(ROOT)

print("Files found:", len(files_df))
display(files_df.sort_values("size_mb", ascending=False).head(50))

files_df.to_csv(FILES_CSV, index=False, encoding="utf-8-sig")
print("Saved:", FILES_CSV)

Files found: 10


,path,relative_path,parent,name,ext,size_mb,size_bytes,modified
4,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series\ts_kkt_id_okved_47.11_r...,dataloader_time_series,ts_kkt_id_okved_47.11_region_77_impute_True.csv,.csv,1024.720,1074496591,2025-11-21 11:34:11
8,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,db_output\okved_47.11_region_77_2023_month_10-...,db_output,okved_47.11_region_77_2023_month_10-12_code_3_...,.csv,742.099,778146740,2025-11-21 11:26:58
5,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series\ts_kkt_id_okved_47.19_r...,dataloader_time_series,ts_kkt_id_okved_47.19_region_77_impute_True.csv,.csv,451.602,473538706,2025-11-21 11:27:41
9,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,db_output\okved_47.19_region_77_2023_month_10-...,db_output,okved_47.19_region_77_2023_month_10-12_code_3_...,.csv,321.633,337256651,2025-11-21 11:26:57
6,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series\ts_stores_okved_47.11_r...,dataloader_time_series,ts_stores_okved_47.11_region_77_impute_True.csv,.csv,244.942,256840714,2025-11-21 11:36:08
7,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series\ts_stores_okved_47.19_r...,dataloader_time_series,ts_stores_okved_47.19_region_77_impute_True.csv,.csv,133.102,139567318,2025-11-21 11:28:29
0,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated\agg_kkt_okved_47.11_regi...,dataloader_aggregated,agg_kkt_okved_47.11_region_77_impute_True.csv,.csv,78.621,82440020,2025-11-21 11:43:39
1,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated\agg_kkt_okved_47.19_regi...,dataloader_aggregated,agg_kkt_okved_47.19_region_77_impute_True.csv,.csv,35.495,37218821,2025-11-21 11:32:05
2,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated\agg_store_okved_47.11_re...,dataloader_aggregated,agg_store_okved_47.11_region_77_impute_True.csv,.csv,1.819,1907745,2025-11-21 11:43:39
3,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated\agg_store_okved_47.19_re...,dataloader_aggregated,agg_store_okved_47.19_region_77_impute_True.csv,.csv,1.021,1070099,2025-11-21 11:32:05


Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data_audit_outputs\kkt_files_inventory.csv


In [4]:
print("=" * 120)
print("EXTENSION SUMMARY")
print("=" * 120)

ext_summary = (
    files_df
    .groupby("ext", as_index=False)
    .agg(
        n_files=("path", "count"),
        total_mb=("size_mb", "sum"),
        max_mb=("size_mb", "max"),
        min_mb=("size_mb", "min"),
    )
    .sort_values("total_mb", ascending=False)
)

display(ext_summary)

print("=" * 120)
print("FOLDER SUMMARY")
print("=" * 120)

folder_summary = (
    files_df
    .groupby("parent", as_index=False)
    .agg(
        n_files=("path", "count"),
        total_mb=("size_mb", "sum"),
        max_mb=("size_mb", "max"),
    )
    .sort_values("total_mb", ascending=False)
)

display(folder_summary.head(100))

ext_summary.to_csv(OUT_DIR / "extension_summary.csv", index=False, encoding="utf-8-sig")
folder_summary.to_csv(OUT_DIR / "folder_summary.csv", index=False, encoding="utf-8-sig")

EXTENSION SUMMARY


,ext,n_files,total_mb,max_mb,min_mb
0,.csv,10,3035.054,1024.72,1.021


FOLDER SUMMARY


,parent,n_files,total_mb,max_mb
1,dataloader_time_series,4,1854.366,1024.720
2,db_output,2,1063.732,742.099
0,dataloader_aggregated,4,116.956,78.621


In [5]:
# ============================================================
# Lightweight readers
# ============================================================

KKT_KEYWORDS = [
    "kkt", "ккт",
    "cash", "cashbox", "cashier",
    "receipt", "check", "cheque", "чек",
    "fn", "фн", "fiscal", "фиск",
    "ofd", "офд",
    "shift", "смен",
    "operation", "операц",
    "amount", "sum", "total", "price", "сум", "итог", "стоим",
    "inn", "инн",
    "rn", "рег", "reg",
    "serial", "завод",
    "date", "time", "дата", "время",
    "tax", "vat", "ндс",
    "payment", "оплат",
    "item", "товар", "позици",
    "buyer", "seller", "продав", "покуп",
    "error", "ошиб",
    "anomaly", "аном",
]


def keyword_score(columns, filename=""):
    text = " ".join([str(c).lower() for c in columns]) + " " + filename.lower()
    hits = [kw for kw in KKT_KEYWORDS if kw in text]
    return len(hits), hits


def try_read_csv_sample(path: Path, nrows=10):
    # Try several separators/encodings.
    encodings = ["utf-8", "utf-8-sig", "cp1251", "windows-1251"]
    seps = [",", ";", "\t", "|"]

    last_error = None

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(
                    path,
                    nrows=nrows,
                    sep=sep,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",
                )
                # If only one column but separator unlikely, continue.
                if df.shape[1] == 1 and sep != ",":
                    continue
                return df, {
                    "reader": "pd.read_csv",
                    "encoding": enc,
                    "sep": sep,
                    "error": None,
                }
            except Exception as e:
                last_error = repr(e)

    return None, {
        "reader": "pd.read_csv",
        "encoding": None,
        "sep": None,
        "error": last_error,
    }


def try_read_csv_gz_sample(path: Path, nrows=10):
    encodings = ["utf-8", "utf-8-sig", "cp1251", "windows-1251"]
    seps = [",", ";", "\t", "|"]

    last_error = None

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(
                    path,
                    compression="gzip",
                    nrows=nrows,
                    sep=sep,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",
                )
                if df.shape[1] == 1 and sep != ",":
                    continue
                return df, {
                    "reader": "pd.read_csv_gzip",
                    "encoding": enc,
                    "sep": sep,
                    "error": None,
                }
            except Exception as e:
                last_error = repr(e)

    return None, {
        "reader": "pd.read_csv_gzip",
        "encoding": None,
        "sep": None,
        "error": last_error,
    }


def try_read_json_sample(path: Path, nrows=10):
    # First try JSONL.
    try:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= nrows:
                    break
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        if rows:
            return pd.json_normalize(rows), {
                "reader": "jsonl_utf8",
                "error": None,
            }
    except Exception as e1:
        err1 = repr(e1)

    # Try ordinary JSON.
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            df = pd.json_normalize(obj[:nrows])
        elif isinstance(obj, dict):
            # If dict has a list inside, use the first list.
            list_key = None
            for k, v in obj.items():
                if isinstance(v, list):
                    list_key = k
                    break

            if list_key is not None:
                df = pd.json_normalize(obj[list_key][:nrows])
            else:
                df = pd.json_normalize([obj])
        else:
            df = pd.DataFrame({"value": [str(obj)]})

        return df, {
            "reader": "json_utf8",
            "error": None,
        }

    except Exception as e2:
        return None, {
            "reader": "json",
            "error": f"jsonl={err1}; json={repr(e2)}",
        }


def try_read_json_gz_sample(path: Path, nrows=10):
    try:
        rows = []
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= nrows:
                    break
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        if rows:
            return pd.json_normalize(rows), {
                "reader": "jsonl_gzip_utf8",
                "error": None,
            }
    except Exception as e:
        return None, {
            "reader": "jsonl_gzip",
            "error": repr(e),
        }


def try_read_parquet_sample(path: Path, nrows=10):
    try:
        # read only first rows after metadata; this may still read file chunks but usually okay.
        df = pd.read_parquet(path)
        if len(df) > nrows:
            df = df.head(nrows)
        return df, {
            "reader": "pd.read_parquet",
            "error": None,
        }
    except Exception as e:
        return None, {
            "reader": "pd.read_parquet",
            "error": repr(e),
        }


def try_read_excel_sample(path: Path, nrows=10):
    try:
        xl = pd.ExcelFile(path)
        sheet = xl.sheet_names[0]
        df = pd.read_excel(path, sheet_name=sheet, nrows=nrows)
        return df, {
            "reader": "pd.read_excel",
            "sheet": sheet,
            "sheets": xl.sheet_names,
            "error": None,
        }
    except Exception as e:
        return None, {
            "reader": "pd.read_excel",
            "error": repr(e),
        }


def read_sample(path: Path, nrows=10):
    ext = detect_ext(path)

    if ext == ".csv":
        return try_read_csv_sample(path, nrows=nrows)

    if ext == ".csv.gz":
        return try_read_csv_gz_sample(path, nrows=nrows)

    if ext in [".json", ".jsonl", ".ndjson"]:
        return try_read_json_sample(path, nrows=nrows)

    if ext in [".json.gz", ".jsonl.gz"]:
        return try_read_json_gz_sample(path, nrows=nrows)

    if ext in [".parquet", ".pq"]:
        return try_read_parquet_sample(path, nrows=nrows)

    if ext in [".xlsx", ".xls"]:
        return try_read_excel_sample(path, nrows=nrows)

    if ext in [".txt", ".log"]:
        return try_read_csv_sample(path, nrows=nrows)

    return None, {
        "reader": "unsupported",
        "error": f"unsupported extension {ext}",
    }

In [6]:
# ============================================================
# Scan all files with samples
# ============================================================

sample_rows = []

# Можно ограничить для первого запуска:
# max_files = 100
max_files = None

files_iter = files_df.sort_values("size_mb", ascending=False)
if max_files is not None:
    files_iter = files_iter.head(max_files)

for i, row in files_iter.iterrows():
    path = Path(row["path"])
    rel = row["relative_path"]
    ext = row["ext"]

    print("\n" + "=" * 120)
    print(f"[{len(sample_rows)+1}] {rel}")
    print(f"size={row['size_mb']} MB ext={ext}")
    print("=" * 120)

    df_sample, meta = read_sample(path, nrows=10)

    info = {
        "relative_path": rel,
        "path": str(path),
        "parent": row["parent"],
        "name": row["name"],
        "ext": ext,
        "size_mb": row["size_mb"],
        "reader": meta.get("reader"),
        "encoding": meta.get("encoding"),
        "sep": meta.get("sep"),
        "sheet": meta.get("sheet"),
        "read_error": meta.get("error"),
        "n_sample_rows": None,
        "n_sample_cols": None,
        "columns": None,
        "keyword_score": 0,
        "keyword_hits": [],
        "candidate_role": None,
    }

    if df_sample is None:
        print("[READ FAILED]", meta.get("error"))
        sample_rows.append(info)
        continue

    info["n_sample_rows"] = int(df_sample.shape[0])
    info["n_sample_cols"] = int(df_sample.shape[1])
    info["columns"] = list(map(str, df_sample.columns))

    score, hits = keyword_score(df_sample.columns, filename=row["name"])
    info["keyword_score"] = score
    info["keyword_hits"] = hits

    # Heuristic role.
    col_text = " ".join(info["columns"]).lower()
    name_text = row["name"].lower()
    full_text = col_text + " " + name_text + " " + row["parent"].lower()

    if any(x in full_text for x in ["receipt", "check", "cheque", "чек", "kkt", "ккт", "fiscal", "фиск"]):
        info["candidate_role"] = "likely_receipts_or_kkt"
    elif any(x in full_text for x in ["item", "товар", "position", "позици"]):
        info["candidate_role"] = "likely_receipt_items"
    elif any(x in full_text for x in ["device", "cashbox", "аппарат", "kkt", "ккт", "terminal"]):
        info["candidate_role"] = "likely_devices"
    elif any(x in full_text for x in ["client", "customer", "inn", "инн", "org", "company"]):
        info["candidate_role"] = "likely_entities"
    elif score >= 3:
        info["candidate_role"] = "possibly_relevant"
    else:
        info["candidate_role"] = "unknown"

    print("reader:", info["reader"], "encoding:", info["encoding"], "sep:", info["sep"], "sheet:", info["sheet"])
    print("shape sample:", df_sample.shape)
    print("keyword_score:", score, "hits:", hits)
    print("candidate_role:", info["candidate_role"])
    print("columns:")
    for c in df_sample.columns:
        print(" -", c)

    display(df_sample.head(10))

    # Save sample to CSV.
    sample_file = SAMPLES_DIR / f"{safe_name(rel)}__sample.csv"
    try:
        df_sample.to_csv(sample_file, index=False, encoding="utf-8-sig")
        info["sample_file"] = str(sample_file)
    except Exception as e:
        info["sample_file"] = None
        info["sample_save_error"] = repr(e)

    sample_rows.append(info)

samples_df = pd.DataFrame(sample_rows)
samples_csv = OUT_DIR / "kkt_files_samples_inventory.csv"
samples_df.to_csv(samples_csv, index=False, encoding="utf-8-sig")

print("\nSaved:", samples_csv)
display(samples_df.sort_values(["keyword_score", "size_mb"], ascending=[False, False]).head(100))


[1] dataloader_time_series\ts_kkt_id_okved_47.11_region_77_impute_True.csv
size=1024.72 MB ext=.csv
reader: pd.read_csv encoding: utf-8 sep: , sheet: None
shape sample: (10, 22)
keyword_score: 8 hits: ['kkt', 'cash', 'sum', 'total', 'price', 'reg', 'date', 'item']
candidate_role: likely_receipts_or_kkt
columns:
 - kkt_id
 - date
 - total_sum
 - avg_sum
 - std_sum
 - skew_sum
 - max_sum
 - min_sum
 - median_sum
 - total_cash
 - total_ecash
 - n_bills
 - n_items
 - total_items
 - trade_store_hash
 - trade_object_hash
 - ecash_fraction
 - avg_price_item
 - avg_price_piece
 - avg_cash
 - avg_ecash
 - ecash_bills_fraction


,kkt_id,date,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,total_cash,...,n_items,total_items,trade_store_hash,trade_object_hash,ecash_fraction,avg_price_item,avg_price_piece,avg_cash,avg_ecash,ecash_bills_fraction
0,00079d6947ea9d8e7fb427f3d478c75e,2023-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
1,00079d6947ea9d8e7fb427f3d478c75e,2023-10-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
2,00079d6947ea9d8e7fb427f3d478c75e,2023-10-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
3,00079d6947ea9d8e7fb427f3d478c75e,2023-10-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
4,00079d6947ea9d8e7fb427f3d478c75e,2023-10-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
5,00079d6947ea9d8e7fb427f3d478c75e,2023-10-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
6,00079d6947ea9d8e7fb427f3d478c75e,2023-10-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
7,00079d6947ea9d8e7fb427f3d478c75e,2023-10-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
8,00079d6947ea9d8e7fb427f3d478c75e,2023-10-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
9,00079d6947ea9d8e7fb427f3d478c75e,2023-10-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0



[2] db_output\okved_47.11_region_77_2023_month_10-12_code_3_operation_1_v1.csv
size=742.099 MB ext=.csv
reader: pd.read_csv encoding: utf-8 sep: , sheet: None
shape sample: (10, 27)
keyword_score: 9 hits: ['kkt', 'cash', 'operation', 'sum', 'total', 'reg', 'date', 'tax', 'item']
candidate_role: likely_receipts_or_kkt
columns:
 - kkt_id
 - date
 - region
 - total_sum
 - avg_sum
 - std_sum
 - skew_sum
 - max_sum
 - min_sum
 - median_sum
 - total_cash
 - total_ecash
 - total_prepaid
 - total_credit
 - total_provision
 - n_bills
 - n_cash_bills
 - n_ecash_bills
 - n_items
 - total_items
 - n_taxation_types
 - okved2
 - okved_group
 - type
 - category
 - trade_store_hash
 - trade_object_hash


,kkt_id,date,region,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,...,n_ecash_bills,n_items,total_items,n_taxation_types,okved2,okved_group,type,category,trade_store_hash,trade_object_hash
0,00079d6947ea9d8e7fb427f3d478c75e,2023-12-08,77,3.00,3.000000,0.000000,NaN,3.00,3.00,3.000,...,0,1,1.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
1,00079d6947ea9d8e7fb427f3d478c75e,2023-12-15,77,33651.70,494.877941,580.649920,3.059434,3720.10,3.00,259.815,...,34,334,424.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
2,00079d6947ea9d8e7fb427f3d478c75e,2023-12-16,77,39556.65,648.469672,758.354135,1.939349,3586.93,3.00,344.000,...,40,307,409.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
3,00079d6947ea9d8e7fb427f3d478c75e,2023-12-17,77,38251.82,579.573030,546.617066,1.535047,2584.00,43.00,438.000,...,16,256,311.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
4,00079d6947ea9d8e7fb427f3d478c75e,2023-12-18,77,70028.43,457.702157,549.172279,4.017840,4819.71,14.99,300.000,...,132,558,696.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
5,00079d6947ea9d8e7fb427f3d478c75e,2023-12-19,77,55.89,55.890000,0.000000,NaN,55.89,55.89,55.890,...,1,1,1.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
6,00079d6947ea9d8e7fb427f3d478c75e,2023-12-20,77,19875.83,432.083261,483.538405,1.677208,1964.93,11.39,260.775,...,39,152,214.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
7,00079d6947ea9d8e7fb427f3d478c75e,2023-12-21,77,46426.40,546.192941,626.000516,3.181687,4398.25,3.00,326.960,...,71,331,422.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
8,00079d6947ea9d8e7fb427f3d478c75e,2023-12-22,77,50960.45,772.128030,1019.417169,2.411603,5259.71,14.00,340.915,...,57,265,399.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
9,00079d6947ea9d8e7fb427f3d478c75e,2023-12-23,77,55061.30,688.266250,817.136370,2.683436,4987.10,3.99,365.730,...,75,307,375.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...



[3] dataloader_time_series\ts_kkt_id_okved_47.19_region_77_impute_True.csv
size=451.602 MB ext=.csv
reader: pd.read_csv encoding: utf-8 sep: , sheet: None
shape sample: (10, 22)
keyword_score: 8 hits: ['kkt', 'cash', 'sum', 'total', 'price', 'reg', 'date', 'item']
candidate_role: likely_receipts_or_kkt
columns:
 - kkt_id
 - date
 - total_sum
 - avg_sum
 - std_sum
 - skew_sum
 - max_sum
 - min_sum
 - median_sum
 - total_cash
 - total_ecash
 - n_bills
 - n_items
 - total_items
 - trade_store_hash
 - trade_object_hash
 - ecash_fraction
 - avg_price_item
 - avg_price_piece
 - avg_cash
 - avg_ecash
 - ecash_bills_fraction


...

,avg_cash__length,avg_ecash__length,avg_price_item__length,avg_price_piece__length,avg_sum__length,cash_fraction__length,ecash_fraction__length,max_sum__length,min_sum__length,n_bills__length,...,total_cash__standard_deviation,total_ecash__sum_values,total_ecash__standard_deviation,total_items__length,total_other__length,total_sum__sum_values,total_sum__standard_deviation,total_sum__length,total_sum__cash_fraction,total_sum__ecash_fraction
0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,9287451.32,13273.793009,92.0,92.0,9287451.32,13273.793009,92.0,0.000000,1.000000
1,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,9953.369940,11222434.40,31223.117641,92.0,92.0,14254359.40,38163.832575,92.0,0.207896,0.787298
2,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,2906347.62,11658.710333,92.0,92.0,2955896.81,11695.388094,92.0,0.000000,0.983237
3,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,886207.41,14540.801082,92.0,92.0,886207.41,14540.801082,92.0,0.000000,1.000000
4,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,3362.482693,433268.42,17348.939333,92.0,92.0,504344.46,20195.074438,92.0,0.140928,0.859072
5,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,4792.488121,962749.51,12617.265305,92.0,92.0,1318442.06,16549.026971,92.0,0.269782,0.730218
6,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,43135.098383,6013361.60,100337.051533,92.0,92.0,6483358.09,124856.163986,92.0,0.064165,0.927507
7,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,13902.468965,4705184.00,15440.147934,92.0,92.0,8262270.00,26325.964350,92.0,0.430522,0.569478
8,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,19706.981761,11186998.61,49604.205673,92.0,92.0,15070786.08,67321.247861,92.0,0.257597,0.742297
9,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,7533.552454,5382192.07,21905.960117,92.0,92.0,6896093.66,28324.822608,92.0,0.219240,0.780470



[10] dataloader_aggregated\agg_store_okved_47.19_region_77_impute_True.csv
size=1.021 MB ext=.csv
reader: pd.read_csv encoding: utf-8 sep: , sheet: None
shape sample: (10, 24)
keyword_score: 6 hits: ['cash', 'sum', 'total', 'price', 'reg', 'item']
candidate_role: likely_receipt_items
columns:
 - avg_cash__length
 - avg_ecash__length
 - avg_price_item__length
 - avg_price_piece__length
 - avg_sum__length
 - cash_fraction__length
 - ecash_fraction__length
 - max_sum__length
 - min_sum__length
 - n_bills__length
 - n_cash_bills__length
 - n_ecash_bills__length
 - n_items__length
 - total_cash__sum_values
 - total_cash__standard_deviation
 - total_ecash__sum_values
 - total_ecash__standard_deviation
 - total_items__length
 - total_other__length
 - total_sum__sum_values
 - total_sum__standard_deviation
 - total_sum__length
 - total_sum__cash_fraction
 - total_sum__ecash_fraction


,avg_cash__length,avg_ecash__length,avg_price_item__length,avg_price_piece__length,avg_sum__length,cash_fraction__length,ecash_fraction__length,max_sum__length,min_sum__length,n_bills__length,...,total_cash__standard_deviation,total_ecash__sum_values,total_ecash__standard_deviation,total_items__length,total_other__length,total_sum__sum_values,total_sum__standard_deviation,total_sum__length,total_sum__cash_fraction,total_sum__ecash_fraction
0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,23977.307433,0.000000e+00,0.000000,92.0,92.0,1.750742e+06,2.397731e+04,92.0,1.000000,0.000000
1,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,18673.451259,7.709901e+06,47216.458082,92.0,92.0,1.101964e+07,6.122878e+04,92.0,0.300349,0.699651
2,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,7.122676e+06,8946.965433,92.0,92.0,7.122676e+06,8.946965e+03,92.0,0.000000,1.000000
3,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,1.148766e+07,110041.638810,92.0,92.0,1.148766e+07,1.100416e+05,92.0,0.000000,1.000000
4,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,2.254647e+08,887841.850418,92.0,92.0,4.383723e+08,1.824766e+06,92.0,0.000000,0.514322
5,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,5683.120213,1.422021e+06,16206.372006,92.0,92.0,1.897878e+06,2.129936e+04,92.0,0.250731,0.749269
6,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,269.591513,4.600000e+03,309.803584,92.0,92.0,7.200000e+03,5.439042e+02,92.0,0.361111,0.638889
7,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,2283.172512,1.112751e+06,6308.535002,92.0,92.0,1.337758e+06,7.189046e+03,92.0,0.168197,0.831803
8,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,0.000000,1.983769e+07,46883.324900,92.0,92.0,1.984211e+07,4.692040e+04,92.0,0.000000,0.999778
9,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,92.0,...,3074.483996,7.982060e+05,25888.534900,92.0,92.0,8.905160e+05,2.888672e+04,92.0,0.103659,0.896341



Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data_audit_outputs\kkt_files_samples_inventory.csv


,relative_path,path,parent,name,ext,size_mb,reader,encoding,sep,sheet,read_error,n_sample_rows,n_sample_cols,columns,keyword_score,keyword_hits,candidate_role,sample_file
1,db_output\okved_47.11_region_77_2023_month_10-...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,db_output,okved_47.11_region_77_2023_month_10-12_code_3_...,.csv,742.099,pd.read_csv,utf-8,",",None,None,10,27,"[kkt_id, date, region, total_sum, avg_sum, std...",9,"[kkt, cash, operation, sum, total, reg, date, ...",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
3,db_output\okved_47.19_region_77_2023_month_10-...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,db_output,okved_47.19_region_77_2023_month_10-12_code_3_...,.csv,321.633,pd.read_csv,utf-8,",",None,None,10,27,"[kkt_id, date, region, total_sum, avg_sum, std...",9,"[kkt, cash, operation, sum, total, reg, date, ...",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
0,dataloader_time_series\ts_kkt_id_okved_47.11_r...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series,ts_kkt_id_okved_47.11_region_77_impute_True.csv,.csv,1024.720,pd.read_csv,utf-8,",",None,None,10,22,"[kkt_id, date, total_sum, avg_sum, std_sum, sk...",8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
2,dataloader_time_series\ts_kkt_id_okved_47.19_r...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series,ts_kkt_id_okved_47.19_region_77_impute_True.csv,.csv,451.602,pd.read_csv,utf-8,",",None,None,10,22,"[kkt_id, date, total_sum, avg_sum, std_sum, sk...",8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
4,dataloader_time_series\ts_stores_okved_47.11_r...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series,ts_stores_okved_47.11_region_77_impute_True.csv,.csv,244.942,pd.read_csv,utf-8,",",None,None,10,22,"[trade_store_hash, date, kkt_id, total_sum, ma...",8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
5,dataloader_time_series\ts_stores_okved_47.19_r...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_time_series,ts_stores_okved_47.19_region_77_impute_True.csv,.csv,133.102,pd.read_csv,utf-8,",",None,None,10,22,"[trade_store_hash, date, kkt_id, total_sum, ma...",8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
6,dataloader_aggregated\agg_kkt_okved_47.11_regi...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated,agg_kkt_okved_47.11_region_77_impute_True.csv,.csv,78.621,pd.read_csv,utf-8,",",None,None,10,156,"[avg_ecash__mean, avg_ecash__standard_deviatio...",7,"[kkt, cash, sum, total, price, reg, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
7,dataloader_aggregated\agg_kkt_okved_47.19_regi...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated,agg_kkt_okved_47.19_region_77_impute_True.csv,.csv,35.495,pd.read_csv,utf-8,",",None,None,10,156,"[avg_ecash__mean, avg_ecash__standard_deviatio...",7,"[kkt, cash, sum, total, price, reg, item]",likely_receipts_or_kkt,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
8,dataloader_aggregated\agg_store_okved_47.11_re...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated,agg_store_okved_47.11_region_77_impute_True.csv,.csv,1.819,pd.read_csv,utf-8,",",None,None,10,24,"[avg_cash__length, avg_ecash__length, avg_pric...",6,"[cash, sum, total, price, reg, item]",likely_receipt_items,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
9,dataloader_aggregated\agg_store_okved_47.19_re...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,dataloader_aggregated,agg_store_okved_47.19_region_77_impute_True.csv,.csv,1.021,pd.read_csv,utf-8,",",None,None,10,24,"[avg_cash__length, avg_ecash__length, avg_pric...

In [7]:
# ============================================================
# TXT report
# ============================================================

REPORT_TXT.unlink(missing_ok=True)

write_report("KKT DATA AUDIT REPORT")
write_report("=" * 120)
write_report(f"Created: {now()}")
write_report(f"Root: {ROOT}")
write_report(f"Files found: {len(files_df)}")
write_report("")

write_report("EXTENSION SUMMARY")
write_report("-" * 120)
write_report(ext_summary.to_string(index=False))
write_report("")

write_report("FOLDER SUMMARY")
write_report("-" * 120)
write_report(folder_summary.head(100).to_string(index=False))
write_report("")

write_report("TOP CANDIDATE FILES BY KEYWORD SCORE")
write_report("-" * 120)

cand = samples_df.sort_values(["keyword_score", "size_mb"], ascending=[False, False]).copy()

cols_to_show = [
    "relative_path",
    "size_mb",
    "ext",
    "reader",
    "n_sample_rows",
    "n_sample_cols",
    "keyword_score",
    "keyword_hits",
    "candidate_role",
    "read_error",
]

write_report(cand[cols_to_show].head(100).to_string(index=False))
write_report("")

write_report("DETAILED FILE SAMPLES")
write_report("=" * 120)

for _, r in cand.iterrows():
    write_report("")
    write_report("-" * 120)
    write_report(f"FILE: {r['relative_path']}")
    write_report(f"SIZE MB: {r['size_mb']}")
    write_report(f"EXT: {r['ext']}")
    write_report(f"READER: {r['reader']}")
    write_report(f"CANDIDATE ROLE: {r['candidate_role']}")
    write_report(f"KEYWORD SCORE: {r['keyword_score']}")
    write_report(f"KEYWORD HITS: {r['keyword_hits']}")
    write_report(f"READ ERROR: {r['read_error']}")
    write_report("COLUMNS:")
    cols = r["columns"]
    if isinstance(cols, str):
        write_report(cols)
    elif isinstance(cols, list):
        for c in cols:
            write_report(f" - {c}")
    else:
        write_report(cols)

print("Saved TXT report:", REPORT_TXT)

Saved TXT report: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data_audit_outputs\kkt_file_audit_report.txt


In [8]:
# ============================================================
# Candidate selection
# ============================================================

candidates = samples_df.copy()

# Правило отбора:
# 1. файл читается
# 2. есть хотя бы 2 колонки
# 3. есть KKT/receipt/payment/time/entity keywords
# 4. не слишком маленький
candidates["is_readable"] = candidates["read_error"].isna()
candidates["has_columns"] = candidates["n_sample_cols"].fillna(0) >= 2
candidates["non_tiny"] = candidates["size_mb"].fillna(0) >= 0.001

candidates["take_candidate"] = (
    candidates["is_readable"]
    & candidates["has_columns"]
    & candidates["non_tiny"]
    & (
        (candidates["keyword_score"].fillna(0) >= 2)
        | candidates["candidate_role"].isin([
            "likely_receipts_or_kkt",
            "likely_receipt_items",
            "likely_devices",
            "likely_entities",
            "possibly_relevant",
        ])
    )
)

candidate_df = candidates[candidates["take_candidate"]].copy()
candidate_df = candidate_df.sort_values(["candidate_role", "keyword_score", "size_mb"], ascending=[True, False, False])

candidate_csv = OUT_DIR / "kkt_candidate_files_to_use.csv"
candidate_df.to_csv(candidate_csv, index=False, encoding="utf-8-sig")

print("Candidate files:", len(candidate_df))
print("Saved:", candidate_csv)

display(candidate_df[[
    "relative_path",
    "size_mb",
    "ext",
    "reader",
    "n_sample_rows",
    "n_sample_cols",
    "keyword_score",
    "keyword_hits",
    "candidate_role",
    "columns",
]])

Candidate files: 10
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data_audit_outputs\kkt_candidate_files_to_use.csv


,relative_path,size_mb,ext,reader,n_sample_rows,n_sample_cols,keyword_score,keyword_hits,candidate_role,columns
8,dataloader_aggregated\agg_store_okved_47.11_re...,1.819,.csv,pd.read_csv,10,24,6,"[cash, sum, total, price, reg, item]",likely_receipt_items,"[avg_cash__length, avg_ecash__length, avg_pric..."
9,dataloader_aggregated\agg_store_okved_47.19_re...,1.021,.csv,pd.read_csv,10,24,6,"[cash, sum, total, price, reg, item]",likely_receipt_items,"[avg_cash__length, avg_ecash__length, avg_pric..."
1,db_output\okved_47.11_region_77_2023_month_10-...,742.099,.csv,pd.read_csv,10,27,9,"[kkt, cash, operation, sum, total, reg, date, ...",likely_receipts_or_kkt,"[kkt_id, date, region, total_sum, avg_sum, std..."
3,db_output\okved_47.19_region_77_2023_month_10-...,321.633,.csv,pd.read_csv,10,27,9,"[kkt, cash, operation, sum, total, reg, date, ...",likely_receipts_or_kkt,"[kkt_id, date, region, total_sum, avg_sum, std..."
0,dataloader_time_series\ts_kkt_id_okved_47.11_r...,1024.720,.csv,pd.read_csv,10,22,8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
2,dataloader_time_series\ts_kkt_id_okved_47.19_r...,451.602,.csv,pd.read_csv,10,22,8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
4,dataloader_time_series\ts_stores_okved_47.11_r...,244.942,.csv,pd.read_csv,10,22,8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,"[trade_store_hash, date, kkt_id, total_sum, ma..."
5,dataloader_time_series\ts_stores_okved_47.19_r...,133.102,.csv,pd.read_csv,10,22,8,"[kkt, cash, sum, total, price, reg, date, item]",likely_receipts_or_kkt,"[trade_store_hash, date, kkt_id, total_sum, ma..."
6,dataloader_aggregated\agg_kkt_okved_47.11_regi...,78.621,.csv,pd.read_csv,10,156,7,"[kkt, cash, sum, total, price, reg, item]",likely_receipts_or_kkt,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
7,dataloader_aggregated\agg_kkt_okved_47.19_regi...,35.495,.csv,pd.read_csv,10,156,7,"[kkt, cash, sum, total, price, reg, item]",likely_receipts_or_kkt,"[avg_ecash__mean, avg_ecash__standard_deviatio..."


In [9]:
# ============================================================
# Classify likely file purpose
# ============================================================

def classify_file(columns, rel_path, name):
    text = " ".join(map(str, columns if isinstance(columns, list) else []))
    text = (text + " " + rel_path + " " + name).lower()

    has_time = any(x in text for x in ["date", "time", "datetime", "timestamp", "дата", "время"])
    has_amount = any(x in text for x in ["sum", "amount", "total", "price", "сум", "итог", "стоим", "payment"])
    has_kkt = any(x in text for x in ["kkt", "ккт", "fn", "фн", "fiscal", "фиск", "ofd", "офд", "cash", "касс"])
    has_id = any(x in text for x in ["id", "uuid", "guid", "rn", "reg", "serial", "номер"])
    has_inn = any(x in text for x in ["inn", "инн"])
    has_item = any(x in text for x in ["item", "товар", "позици", "product", "name"])
    has_error = any(x in text for x in ["error", "ошиб", "fail", "status", "state"])

    if has_kkt and has_time and has_amount:
        return "PRIMARY_RECEIPT_EVENTS"
    if has_kkt and has_time:
        return "KKT_DEVICE_EVENTS"
    if has_amount and has_time:
        return "TRANSACTION_EVENTS"
    if has_item:
        return "RECEIPT_ITEMS"
    if has_kkt and has_id:
        return "DEVICE_REFERENCE"
    if has_inn:
        return "ENTITY_REFERENCE"
    if has_error:
        return "ERROR_OR_STATUS_LOG"
    return "OTHER"


candidate_df["inferred_purpose"] = candidate_df.apply(
    lambda r: classify_file(r["columns"], r["relative_path"], r["name"]),
    axis=1,
)

purpose_summary = (
    candidate_df
    .groupby("inferred_purpose", as_index=False)
    .agg(
        n_files=("path", "count"),
        total_mb=("size_mb", "sum"),
        max_mb=("size_mb", "max"),
    )
    .sort_values("total_mb", ascending=False)
)

display(purpose_summary)

candidate_df = candidate_df.sort_values(["inferred_purpose", "size_mb"], ascending=[True, False])
display(candidate_df[[
    "inferred_purpose",
    "relative_path",
    "size_mb",
    "ext",
    "n_sample_cols",
    "keyword_score",
    "columns",
]])

candidate_df.to_csv(OUT_DIR / "kkt_candidate_files_classified.csv", index=False, encoding="utf-8-sig")

,inferred_purpose,n_files,total_mb,max_mb
0,PRIMARY_RECEIPT_EVENTS,6,2918.098,1024.720
1,RECEIPT_ITEMS,4,116.956,78.621


,inferred_purpose,relative_path,size_mb,ext,n_sample_cols,keyword_score,columns
0,PRIMARY_RECEIPT_EVENTS,dataloader_time_series\ts_kkt_id_okved_47.11_r...,1024.720,.csv,22,8,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
1,PRIMARY_RECEIPT_EVENTS,db_output\okved_47.11_region_77_2023_month_10-...,742.099,.csv,27,9,"[kkt_id, date, region, total_sum, avg_sum, std..."
2,PRIMARY_RECEIPT_EVENTS,dataloader_time_series\ts_kkt_id_okved_47.19_r...,451.602,.csv,22,8,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
3,PRIMARY_RECEIPT_EVENTS,db_output\okved_47.19_region_77_2023_month_10-...,321.633,.csv,27,9,"[kkt_id, date, region, total_sum, avg_sum, std..."
4,PRIMARY_RECEIPT_EVENTS,dataloader_time_series\ts_stores_okved_47.11_r...,244.942,.csv,22,8,"[trade_store_hash, date, kkt_id, total_sum, ma..."
5,PRIMARY_RECEIPT_EVENTS,dataloader_time_series\ts_stores_okved_47.19_r...,133.102,.csv,22,8,"[trade_store_hash, date, kkt_id, total_sum, ma..."
6,RECEIPT_ITEMS,dataloader_aggregated\agg_kkt_okved_47.11_regi...,78.621,.csv,156,7,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
7,RECEIPT_ITEMS,dataloader_aggregated\agg_kkt_okved_47.19_regi...,35.495,.csv,156,7,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
8,RECEIPT_ITEMS,dataloader_aggregated\agg_store_okved_47.11_re...,1.819,.csv,24,6,"[avg_cash__length, avg_ecash__length, avg_pric..."
9,RECEIPT_ITEMS,dataloader_aggregated\agg_store_okved_47.19_re...,1.021,.csv,24,6,"[avg_cash__length, avg_ecash__length, avg_pric..."
